## prompt para o Genie Code
Atue como um Engenheiro de Dados Sênior e Especialista em Databricks. Preciso que você escreva um script Python utilizando Spark (em um notebook Databricks) para realizar a Ingestão da Camada Bronze de um pipeline de dados.

Adapte o script utilizado aula_2_3_ingestao_bronze.ipynb. A ingestão continua a mesma mas é necessário fazer alguns ajustes:
* o primeiro é que podem chegar vários arquivos .csv no Volumes /Volumes/workspace/raw/bronze/input/, então é necessários mover eles após a finalização da ingestão para uma pasta /Volumes/workspace/raw/bronze/processados/, 
* E a tabela final deve ser salva em um catálogo novo cahamdo de prod, garante a criação deste catalogo também.
* O database estava no raw, vamos criar o database bronze. `CREATE DATABASE IF NOT EXISTS prod.bronze`
* a tabela precisa ser particionada, deve-ser criar uma coluna data_particao no formato yyyy-MM-dd a partir da coluna data_hora

# Ingestao Bronze - Camada Prod

Pipeline de ingestao da camada Bronze no catalogo **prod** com leitura de multiplos CSVs, particionamento por `data_particao` e movimentacao de arquivos processados.

In [0]:
# ========================================
# PASSO 1: Criar Catalogo e Database
# ========================================

# Criar o catalogo 'prod'
spark.sql("""
    CREATE CATALOG IF NOT EXISTS prod
    COMMENT 'Catalogo de producao para o pipeline de dados AluMax'
""")

print("Catalogo 'prod' criado/verificado com sucesso!")

# Criar o database 'prod.bronze'
spark.sql("""
    CREATE DATABASE IF NOT EXISTS prod.bronze
    COMMENT 'Schema bronze para dados brutos do pipeline AluMax'
""")

print("Database 'prod.bronze' criado/verificado com sucesso!")

In [0]:
# ========================================
# PASSO 2: Definir Schema Explicito
# ========================================

from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Schema explicito para evitar inferencia e garantir consistencia
schema_bronze = StructType([
    StructField("id_interacao", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("status", StringType(), True),
    StructField("data_hora", TimestampType(), True),
    StructField("payload_detalhes", StringType(), True)
])

print("Schema explicito definido com sucesso!")
print("\nEstrutura do Schema:")
for field in schema_bronze.fields:
    print(f"  - {field.name}: {field.dataType}")

In [0]:
# ========================================
# PASSO 3: Listar e Ler todos os CSVs do diretorio input
# ========================================

import os

# Diretorios de entrada e saidida
input_dir = "/Volumes/workspace/raw/bronze/input/"
processados_dir = "/Volumes/workspace/raw/bronze/processados/"

# Listar todos os arquivos .csv no diretorio de entrada
csv_files = [f.path for f in dbutils.fs.ls(input_dir) if f.name.endswith(".csv")]

if not csv_files:
    print("Nenhum arquivo .csv encontrado no diretorio de input!")
else:
    print(f"Arquivos CSV encontrados: {len(csv_files)}")
    for f in csv_files:
        print(f"  - {f}")

    # Leitura de TODOS os CSVs do diretorio (Spark le multiplos arquivos de uma vez)
    df_source = spark.read.format("csv") \
        .schema(schema_bronze) \
        .option("header", "true") \
        .option("sep", ";") \
        .option("encoding", "UTF-8") \
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
        .load(input_dir)

    print(f"\nTotal de registros lidos: {df_source.count()}")
    print("\nPreview dos dados de origem:")
    df_source.show(3, truncate=False)

In [0]:
# ========================================
# PASSO 4: Adicionar Metadados e Coluna de Particao
# ========================================

from pyspark.sql.functions import current_timestamp, col, date_format

# Adicionar colunas de auditoria e particao
df_bronze = df_source \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("data_particao", date_format(col("data_hora"), "yyyy-MM-dd"))

print("Metadados de auditoria adicionados!")
print("  - _ingestion_timestamp: Data/hora do processamento")
print("  - _source_file: Caminho do arquivo de origem")
print("  - data_particao: Coluna de particao (yyyy-MM-dd)")

print("\nPreview com metadados:")
df_bronze.select("id_interacao", "cliente_id", "data_hora", "data_particao", "_ingestion_timestamp", "_source_file").show(3, truncate=False)

In [0]:
# ========================================
# PASSO 5: Escrever na Tabela Bronze (Delta Lake)
# ========================================

# Nome da tabela gerenciada no catalogo prod
table_name = "prod.bronze.atendimentos_alumax"

# Escrever no Delta Lake com PARTICIONAMENTO e modo APPEND
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("data_particao") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"Dados inseridos na tabela '{table_name}' com sucesso!")
print(f"Formato: Delta Lake (ACID compliant)")
print(f"Particionamento: data_particao (yyyy-MM-dd)")
print(f"Modo: APPEND (incremental)")

In [0]:
# ========================================
# PASSO 6: Mover arquivos processados para a pasta processados
# ========================================

# Garantir que o diretorio de processados existe
try:
    dbutils.fs.ls(processados_dir)
except Exception:
    dbutils.fs.mkdirs(processados_dir)
    print(f"Diretorio criado: {processados_dir}")

# Mover cada arquivo CSV do input para processados
for csv_file in csv_files:
    file_name = csv_file.split("/")[-1]
    dest_path = processados_dir + file_name
    dbutils.fs.mv(csv_file, dest_path)
    print(f"Movido: {file_name} -> {dest_path}")

print(f"\nTotal de arquivos movidos: {len(csv_files)}")
print("Todos os arquivos foram movidos para a pasta 'processados'!")

In [0]:
# ========================================
# PASSO 7: Otimizar Tabela Bronze (Z-Ordering)
# ========================================

print("Otimizando a tabela Bronze...")

spark.sql(f"""
    OPTIMIZE {table_name}
    ZORDER BY (canal, departamento, status)
""")

print("Otimizacao concluida!")
print("  - Z-Ordering aplicado em: canal, departamento, status")
print("  - Queries filtradas por essas colunas terao melhor performance")

In [0]:
# ========================================
# PASSO 8: Validacao Final
# ========================================

print("VALIDACAO DA TABELA BRONZE")
print("=" * 50)

# 1. Schema da tabela
print("\n1. Schema da Tabela:")
spark.table(table_name).printSchema()

# 2. Total de registros
total_records = spark.table(table_name).count()
print(f"\n2. Total de registros: {total_records:,}")

# 3. Distribuicao por particao
print("\n3. Distribuicao por Particao (data_particao):")
spark.sql(f"""
    SELECT 
        data_particao,
        COUNT(*) as total_registros
    FROM {table_name}
    GROUP BY data_particao
    ORDER BY data_particao DESC
""").show(10)

# 4. Estatisticas por canal
print("\n4. Distribuicao por Canal:")
spark.sql(f"""
    SELECT 
        canal,
        COUNT(*) as total
    FROM {table_name}
    GROUP BY canal
    ORDER BY total DESC
""").show()

# 5. Amostra dos dados
print("\n5. Amostra dos primeiros 5 registros:")
df_sample = spark.table(table_name).limit(5)
display(df_sample)